# 🤖 POSHA Autonomous Cooking Robot — Path Planning
## Robotics Engineering Internship Assignment
**Candidate:** Rachit | SRM IST Kattankulathur (ECE + Data Science, 2026)

---

### Overview
This notebook implements **path planning for the AgileX Piper 6-DOF robotic arm** in the Posha autonomous kitchen robot.

- **Task 1 (Macro Dispense):** Container 5 → Pan 2, Container 1 → Pan 1
- **Task 2 (Micro Dispense):** Spice Pod 1 → Pan 2, Spice Pod 19 → Pan 1

### Approach
- Coordinates extracted from `POSHA_Robotics_Assignment.step` via FreeCAD
- Forward Kinematics via Denavit-Hartenberg parameters (from URDF)
- Inverse Kinematics via Damped Least-Squares Jacobian
- Collision detection via bounding-sphere model
- Visualization via matplotlib 3D

## 1. Install Dependencies

In [ ]:
!pip install numpy matplotlib -q
import numpy as np
import math
import json
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from dataclasses import dataclass, field
from typing import List, Tuple, Optional, Dict
print('✅ All dependencies loaded')

## 2. Workspace Coordinate System

All coordinates extracted from `POSHA_Robotics_Assignment.step` via FreeCAD Python console.

| Component | X (m) | Y (m) | Z (m) |
|-----------|--------|--------|--------|
| Robot Base (PIPER) | 0.000 | 0.000 | 0.000 |
| Container 1 | 0.304 | 0.628 | 0.738 |
| Container 5 | 0.389 | 0.471 | 0.737 |
| Pan 1 | 0.188 | 0.641 | 0.741 |
| Pan 2 | 0.689 | 0.642 | 0.740 |
| Spice Pod 1 | 0.176 | 0.805 | 0.922 |
| Spice Pod 19 | 0.293 | 0.805 | 1.042 |

In [ ]:
MM2M = 0.001

@dataclass
class Pose:
    x: float
    y: float
    z: float
    roll: float = 0.0
    pitch: float = 0.0
    yaw: float = 0.0
    def to_array(self): return np.array([self.x, self.y, self.z])

ROBOT_BASE = Pose(0.0, 0.0, 0.0)

MACRO_CONTAINERS = {
    1: Pose(303.6*MM2M, 627.5*MM2M, 737.6*MM2M),
    2: Pose(498.1*MM2M, 789.4*MM2M, 737.3*MM2M),
    3: Pose(305.7*MM2M, 703.0*MM2M, 737.7*MM2M),
    4: Pose(572.6*MM2M, 638.7*MM2M, 738.2*MM2M),
    5: Pose(389.0*MM2M, 471.1*MM2M, 737.0*MM2M),
    6: Pose(389.3*MM2M, 697.1*MM2M, 737.3*MM2M),
    7: Pose(574.0*MM2M, 564.1*MM2M, 737.4*MM2M),
}

PAN_CENTERS = {
    1: Pose(187.9*MM2M, 640.9*MM2M, 740.8*MM2M),
    2: Pose(689.0*MM2M, 641.9*MM2M, 740.3*MM2M),
}

PAN_RADIUS = 0.130
SAFE_Z_CLEARANCE = 0.250
CONTAINER_HEIGHT = 0.120
CONTAINER_LIP_OFFSET = 0.005
GRIPPER_APPROACH_CLEARANCE = 0.080
PAN_DISPENSE_HEIGHT_ABOVE = 0.100
COLLISION_RADIUS = 0.080

print('✅ Workspace coordinates loaded')
print(f'   Container 5: {MACRO_CONTAINERS[5]}')
print(f'   Pan 2:       {PAN_CENTERS[2]}')

## 3. AgileX Piper Kinematics

### DH Parameters (from piper_description.urdf)

| Joint | a (m) | α (rad) | d (m) | θ offset |
|-------|--------|---------|-------|----------|
| J1 | 0.000 | 0.000 | 0.123 | 0.000 |
| J2 | 0.285 | -π/2 | 0.000 | -0.136 |
| J3 | 0.022 | 0.000 | 0.251 | -1.794 |
| J4 | 0.000 | π/2 | 0.000 | 0.000 |
| J5 | 0.000 | -π/2 | 0.091 | 0.000 |
| J6 | 0.000 | π/2 | 0.136 | 0.000 |

### Transformation Matrix
$$T_i = \begin{bmatrix} \cos\theta & -\sin\theta\cos\alpha & \sin\theta\sin\alpha & a\cos\theta \\ \sin\theta & \cos\theta\cos\alpha & -\cos\theta\sin\alpha & a\sin\theta \\ 0 & \sin\alpha & \cos\alpha & d \\ 0 & 0 & 0 & 1 \end{bmatrix}$$

In [ ]:
class PiperKinematics:
    JOINT_LIMITS = {
        'j1': (-2.618, 2.618), 'j2': (0.0, 3.14),
        'j3': (-2.967, 0.0),   'j4': (-1.745, 1.745),
        'j5': (-1.22, 1.22),   'j6': (-2.094, 2.094),
    }
    DH_PARAMS = [
        [0.0,     0.0,          0.123,  0.0    ],
        [0.28503, -math.pi/2,   0.0,   -0.1359 ],
        [0.021984, 0.0,         0.25075,-1.7939],
        [0.0,     math.pi/2,    0.0,    0.0    ],
        [0.0,    -math.pi/2,    0.091,  0.0    ],
        [0.0,     math.pi/2,    0.1358, 0.0    ],
    ]

    @staticmethod
    def dh_transform(a, alpha, d, theta):
        ct, st = math.cos(theta), math.sin(theta)
        ca, sa = math.cos(alpha), math.sin(alpha)
        return np.array([
            [ct, -st*ca,  st*sa, a*ct],
            [st,  ct*ca, -ct*sa, a*st],
            [0.0, sa,     ca,    d   ],
            [0.0, 0.0,    0.0,   1.0 ]
        ])

    @classmethod
    def forward_kinematics(cls, joint_angles):
        T = np.eye(4)
        for i, (a, alpha, d, theta_offset) in enumerate(cls.DH_PARAMS):
            theta = joint_angles[i] + theta_offset
            T = T @ cls.dh_transform(a, alpha, d, theta)
        return T[:3, 3], T[:3, :3]

    @classmethod
    def inverse_kinematics(cls, target, initial_guess=None, max_iter=500, tolerance=1e-4):
        q = initial_guess or [0.0, math.pi/4, -math.pi/4, 0.0, math.pi/4, 0.0]
        q = list(q)
        target_pos = target.to_array()
        for _ in range(max_iter):
            pos, _ = cls.forward_kinematics(q)
            error = target_pos - pos
            if np.linalg.norm(error) < tolerance:
                return cls._clamp(q), True
            J = np.zeros((3, 6))
            for j in range(6):
                qp = q.copy(); qp[j] += 1e-6
                pp, _ = cls.forward_kinematics(qp)
                J[:, j] = (pp - pos) / 1e-6
            dq = J.T @ np.linalg.solve(J @ J.T + 0.01**2 * np.eye(3), error)
            q = cls._clamp([q[i] + dq[i] for i in range(6)])
        return cls._clamp(q), False

    @classmethod
    def _clamp(cls, q):
        lims = list(cls.JOINT_LIMITS.values())
        return [max(lims[i][0], min(lims[i][1], q[i])) for i in range(6)]

    @classmethod
    def check_reachability(cls, target):
        return np.linalg.norm(target.to_array()) <= 0.76

# Test FK
home = [0.0, 0.3, -0.5, 0.0, 0.4, 0.0]
pos, rot = PiperKinematics.forward_kinematics(home)
print(f'✅ FK at home: EE position = ({pos[0]:.3f}, {pos[1]:.3f}, {pos[2]:.3f}) m')
print(f'✅ Rotation matrix shape: {rot.shape}')

## 4. Collision Detection

In [ ]:
class CollisionChecker:
    def __init__(self):
        self.obstacles = [
            {'name': 'Stove',     'center': np.array([46.2,866.9,-187.2])*MM2M, 'radius': 0.25},
            {'name': 'SpiceRack', 'center': np.array([234.0,804.9,1042.5])*MM2M,'radius': 0.18},
            {'name': 'Pan1_rim',  'center': PAN_CENTERS[1].to_array(), 'radius': PAN_RADIUS+0.02},
            {'name': 'Pan2_rim',  'center': PAN_CENTERS[2].to_array(), 'radius': PAN_RADIUS+0.02},
        ]

    def check_point(self, point, ignore=''):
        for obs in self.obstacles:
            if obs['name'] == ignore: continue
            if np.linalg.norm(point - obs['center']) < obs['radius'] + COLLISION_RADIUS:
                return False, obs['name']
        return True, ''

    def check_path(self, start, end, n=20, ignore=''):
        for t in np.linspace(0,1,n):
            ok, name = self.check_point(start + t*(end-start), ignore)
            if not ok: return False, name
        return True, ''

cc = CollisionChecker()
ok, blocker = cc.check_path(MACRO_CONTAINERS[5].to_array(), PAN_CENTERS[2].to_array())
print(f'✅ Collision check (C5→P2 direct): collision_free={ok}, blocker="{blocker}"')
print(f'   → Detour waypoint will be inserted automatically')

## 5. Coordinate Frames & Transformation Matrices

In [ ]:
print('='*65)
print('COORDINATE FRAMES AND TRANSFORMATION MATRICES')
print('='*65)

# World origin
T_world = np.eye(4)
print('\n[T_world] — Assembly Origin (Identity):')
print(np.round(T_world, 4))

# Container 5 frame
c5 = MACRO_CONTAINERS[5]
T_c5 = np.eye(4); T_c5[:3,3] = c5.to_array()
print('\n[T_container5] — Container 5 position frame:')
print(np.round(T_c5, 4))

# Pan 2 frame
p2 = PAN_CENTERS[2]
T_p2 = np.eye(4); T_p2[:3,3] = p2.to_array()
print('\n[T_pan2] — Pan 2 position frame:')
print(np.round(T_p2, 4))

# Transform C5 → P2
T_c5_to_p2 = np.linalg.inv(T_c5) @ T_p2
print('\n[T_c5_to_p2] — Transform Container5 → Pan2:')
print(np.round(T_c5_to_p2, 4))
print(f'\n   Translation: Δx={T_c5_to_p2[0,3]:.3f}m, Δy={T_c5_to_p2[1,3]:.3f}m, Δz={T_c5_to_p2[2,3]:.4f}m')

# EE at home
pos, rot = PiperKinematics.forward_kinematics([0.0, 0.3, -0.5, 0.0, 0.4, 0.0])
T_ee = np.eye(4); T_ee[:3,:3] = rot; T_ee[:3,3] = pos
print('\n[T_ee_home] — End-effector at home position:')
print(np.round(T_ee, 4))

## 6. Task 1: Macro Dispense Path Planning

### Algorithm
```
Step 1: ALIGN   — Move above container at safe Z clearance
Step 2: GRIP    — Descend to 5mm above rear lip; close gripper
Step 3: LIFT    — Raise to safe Z; transit horizontally above pan
Step 4: DISPENSE — Lower to 10cm above pan; tilt 180° to pour
Step 5: RETURN  — Lift and transit back above container
Step 6: PLACE   — Descend and release container at original position
```

In [ ]:
@dataclass
class Waypoint:
    name: str
    pose: Pose
    joint_angles: List[float] = field(default_factory=list)
    gripper_state: str = 'open'
    speed: float = 0.05

def plan_macro_dispense(container_id, pan_id):
    container = MACRO_CONTAINERS[container_id]
    pan = PAN_CENTERS[pan_id]
    cc = CollisionChecker()
    home_j = [0.0, 0.3, -0.5, 0.0, 0.4, 0.0]
    home_pos, _ = PiperKinematics.forward_kinematics(home_j)
    wps, prev = [], home_j.copy()

    def add_wp(name, pose, gripper, speed=0.05):
        q, ok = PiperKinematics.inverse_kinematics(pose, prev)
        prev[:] = q
        wps.append(Waypoint(name, pose, q.copy(), gripper, speed))
        status = '✅' if ok else '⚠️ '
        deg = [round(math.degrees(a),1) for a in q]
        print(f'  {status} {name}')
        print(f'      pos=({pose.x:.3f},{pose.y:.3f},{pose.z:.3f}) joints°={deg}')

    print(f'\n{"="*55}')
    print(f'MACRO DISPENSE: Container {container_id} → Pan {pan_id}')
    print(f'{"="*55}')

    wps.append(Waypoint('HOME', Pose(*home_pos), home_j.copy(), 'open'))

    add_wp(f'STEP1_Align_C{container_id}',
           Pose(container.x, container.y, container.z+CONTAINER_HEIGHT+GRIPPER_APPROACH_CLEARANCE),
           'open', 0.08)
    add_wp(f'STEP2_Grip_C{container_id}',
           Pose(container.x, container.y, container.z+CONTAINER_HEIGHT-CONTAINER_LIP_OFFSET,
                pitch=-math.pi/6),
           'closed', 0.03)

    lift = Pose(container.x, container.y, container.z+CONTAINER_HEIGHT+SAFE_Z_CLEARANCE)
    add_wp(f'STEP3a_Lift_C{container_id}', lift, 'closed', 0.05)

    transit = Pose(pan.x, pan.y, pan.z+PAN_DISPENSE_HEIGHT_ABOVE+SAFE_Z_CLEARANCE)
    ok_col, blocker = cc.check_path(lift.to_array(), transit.to_array())
    if not ok_col:
        print(f'  ⚠️  Collision with {blocker} → inserting detour waypoint')
        mid = Pose((container.x+pan.x)/2, (container.y+pan.y)/2, max(lift.z,transit.z)+0.05)
        add_wp('STEP3b_Detour', mid, 'closed', 0.08)

    add_wp(f'STEP3c_Above_Pan{pan_id}', transit, 'closed', 0.08)
    add_wp(f'STEP4_Dispense_P{pan_id}',
           Pose(pan.x, pan.y, pan.z+PAN_DISPENSE_HEIGHT_ABOVE, pitch=math.pi),
           'closed', 0.02)
    add_wp('STEP5a_Rise', transit, 'closed', 0.05)
    add_wp(f'STEP5b_Return_C{container_id}', lift, 'closed', 0.08)
    add_wp(f'STEP6_Place_C{container_id}',
           Pose(container.x, container.y, container.z+CONTAINER_HEIGHT-CONTAINER_LIP_OFFSET),
           'open', 0.03)
    wps.append(Waypoint('HOME_RETURN', Pose(*home_pos), home_j.copy(), 'open'))

    total_time = sum(
        np.linalg.norm(wps[i].pose.to_array()-wps[i-1].pose.to_array()) /
        max((wps[i].speed+wps[i-1].speed)/2, 0.001)
        for i in range(1, len(wps))
    )
    print(f'\n  📊 Total waypoints: {len(wps)}')
    print(f'  ⏱️  Estimated execution time: {total_time:.1f}s')
    return wps

path_c5_p2 = plan_macro_dispense(container_id=5, pan_id=2)

In [ ]:
path_c1_p1 = plan_macro_dispense(container_id=1, pan_id=1)

## 7. Task 1 Visualization

In [ ]:
fig = plt.figure(figsize=(16, 7))
fig.patch.set_facecolor('#1a1a2e')

# 3D view
ax1 = fig.add_subplot(121, projection='3d')
ax1.set_facecolor('#16213e')
ax1.set_title('Task 1: Macro Dispense — 3D Path', color='white', fontsize=12, fontweight='bold')

# Draw containers
for cid, cp in MACRO_CONTAINERS.items():
    ax1.bar3d(cp.x-0.04, cp.y-0.03, cp.z, 0.08, 0.06, 0.12,
              color='#ffa500', alpha=0.5, shade=True)
    ax1.text(cp.x, cp.y, cp.z+0.14, f'C{cid}', color='orange', fontsize=6, ha='center')

# Draw pans
for pid, pp in PAN_CENTERS.items():
    theta = np.linspace(0, 2*np.pi, 30)
    z_cyl = np.array([pp.z, pp.z+0.06])
    th_g, z_g = np.meshgrid(theta, z_cyl)
    ax1.plot_surface(pp.x+0.13*np.cos(th_g), pp.y+0.13*np.sin(th_g), z_g,
                    color='#4fc3f7', alpha=0.3)
    ax1.text(pp.x, pp.y, pp.z+0.09, f'Pan{pid}', color='#4fc3f7', fontsize=8, ha='center')

# Robot base
ax1.scatter([0],[0],[0], s=150, c='lime', marker='^', zorder=10)
ax1.text(0, 0, 0.05, 'Robot\nBase', color='lime', fontsize=7, ha='center')

# Paths
pts_c5 = np.array([[w.pose.x, w.pose.y, w.pose.z] for w in path_c5_p2])
pts_c1 = np.array([[w.pose.x, w.pose.y, w.pose.z] for w in path_c1_p1])
ax1.plot(pts_c5[:,0], pts_c5[:,1], pts_c5[:,2], 'r-o', lw=2, ms=5, label='C5→Pan2', alpha=0.9)
ax1.plot(pts_c1[:,0], pts_c1[:,1], pts_c1[:,2], 'y-o', lw=2, ms=5, label='C1→Pan1', alpha=0.9)

# Highlight STEP waypoints
for wp in path_c5_p2:
    if 'STEP' in wp.name:
        ax1.scatter([wp.pose.x],[wp.pose.y],[wp.pose.z], s=40, c='red', alpha=0.9, zorder=5)

ax1.set_xlabel('X (m)', color='white', fontsize=8)
ax1.set_ylabel('Y (m)', color='white', fontsize=8)
ax1.set_zlabel('Z (m)', color='white', fontsize=8)
ax1.tick_params(colors='white', labelsize=6)
ax1.xaxis.pane.fill = ax1.yaxis.pane.fill = ax1.zaxis.pane.fill = False
ax1.legend(fontsize=9, labelcolor='white', facecolor='#1a1a2e', framealpha=0.6)

# Top-down
ax2 = fig.add_subplot(122)
ax2.set_facecolor('#16213e')
ax2.set_title('Task 1: Top-Down View (XY Plane)', color='white', fontsize=12, fontweight='bold')

for cid, cp in MACRO_CONTAINERS.items():
    r = plt.Rectangle((cp.x-0.04,cp.y-0.03),0.08,0.06,facecolor='#ffa500',alpha=0.6,edgecolor='white',lw=0.5)
    ax2.add_patch(r)
    ax2.text(cp.x, cp.y, f'C{cid}', ha='center', va='center', fontsize=7, fontweight='bold')

for pid, pp in PAN_CENTERS.items():
    c = plt.Circle((pp.x,pp.y),0.13,facecolor='#4fc3f7',alpha=0.4,edgecolor='white',lw=1)
    ax2.add_patch(c)
    ax2.text(pp.x, pp.y, f'Pan{pid}', ha='center', va='center', fontsize=9, fontweight='bold')

ax2.plot(pts_c5[:,0], pts_c5[:,1], 'r-o', lw=2, ms=5, label='C5→Pan2')
ax2.plot(pts_c1[:,0], pts_c1[:,1], 'y-o', lw=2, ms=5, label='C1→Pan1')
ax2.plot(0, 0, '^', ms=12, color='lime', label='Robot Base', zorder=10)
ax2.set_xlabel('X (m)', color='white')
ax2.set_ylabel('Y (m)', color='white')
ax2.tick_params(colors='white')
ax2.legend(fontsize=9, facecolor='#1a1a2e', labelcolor='white', framealpha=0.7)
ax2.grid(True, color='gray', alpha=0.2)
for sp in ax2.spines.values(): sp.set_edgecolor('gray')

plt.tight_layout()
plt.savefig('task1_macro_path.png', dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()
print('✅ Task 1 visualization complete')

## 8. Task 2: Micro Dispense Path Planning

### Algorithm
```
Step 1: ALIGN   — Approach from front of rack; align with pod
Step 2: GRIP    — Move actuator 2mm from pod surface; close gripper
Step 3: LIFT    — Extract pod from rack; transit above pan
Step 4: DISPENSE — Lower to pan; tilt to pour spice
Step 5: RETURN  — Return pod to exact rack position; release
```

### Spice Pod Layout (from STEP file)
- **Column A (Left):** X = 175.5mm, 10 pods
- **Column B (Right):** X = 292.5mm, 10 pods
- **Y fixed:** 804.9mm (rack depth)
- **Z range:** 921.5mm → 1163.5mm

In [ ]:
_POD_Z = [921.5, 982.0, 1042.5, 1103.0, 1163.5]
_COL_A_X, _COL_B_X, _POD_Y = 175.5, 292.5, 804.9

def build_pod_map():
    pods, pid = {}, 1
    z_ext = sorted(_POD_Z * 2)
    for z in z_ext:
        pods[pid] = Pose(_COL_A_X*MM2M, _POD_Y*MM2M, z*MM2M)
        pid += 1
    for z in z_ext:
        pods[pid] = Pose(_COL_B_X*MM2M, _POD_Y*MM2M, z*MM2M)
        pid += 1
    return pods

SPICE_PODS = build_pod_map()

# Print pod table
print(f'{'Pod':<6} {'X(m)':<10} {'Y(m)':<10} {'Z(m)':<10} {'Column'}')
print('-'*50)
for pid, p in sorted(SPICE_PODS.items()):
    col = 'A (Left)' if abs(p.x - _COL_A_X*MM2M) < 0.001 else 'B (Right)'
    print(f'{pid:<6} {p.x:<10.4f} {p.y:<10.4f} {p.z:<10.4f} {col}')

In [ ]:
SPICE_POD_APPROACH = 0.002
SPICE_POD_DIAMETER = 0.035
SPICE_POD_HEIGHT = 0.060
SPICE_POD_LIFT = 0.120

def plan_micro_dispense(pod_id, pan_id):
    pod = SPICE_PODS[pod_id]
    pan = PAN_CENTERS[pan_id]
    home_j = [0.0, 0.3, -0.5, 0.0, 0.4, 0.0]
    home_pos, _ = PiperKinematics.forward_kinematics(home_j)
    wps, prev = [], home_j.copy()

    def add_wp(name, pose, gripper, speed=0.05):
        q, ok = PiperKinematics.inverse_kinematics(pose, prev)
        prev[:] = q
        wps.append(Waypoint(name, pose, q.copy(), gripper, speed))
        status = '✅' if ok else '⚠️ '
        print(f'  {status} {name} → ({pose.x:.3f},{pose.y:.3f},{pose.z:.3f})')

    print(f'\n{"="*50}')
    print(f'MICRO DISPENSE: Pod {pod_id} → Pan {pan_id}')
    print(f'Pod: ({pod.x:.4f},{pod.y:.4f},{pod.z:.4f})')
    print(f'{"="*50}')

    wps.append(Waypoint('HOME', Pose(*home_pos), home_j.copy(), 'open'))
    add_wp(f'STEP1_Align_Pod{pod_id}',
           Pose(pod.x, pod.y-0.080, pod.z+SPICE_POD_LIFT), 'open', 0.08)
    add_wp(f'STEP1b_PreGrip_Pod{pod_id}',
           Pose(pod.x, pod.y-0.040, pod.z+SPICE_POD_HEIGHT/2), 'open', 0.04)
    grip_pose = Pose(pod.x, pod.y-(SPICE_POD_DIAMETER/2)-SPICE_POD_APPROACH,
                     pod.z+SPICE_POD_HEIGHT/2)
    add_wp(f'STEP2_Grip_Pod{pod_id}', grip_pose, 'closed', 0.02)
    lift_pose = Pose(pod.x, pod.y-0.080, pod.z+SPICE_POD_LIFT+SAFE_Z_CLEARANCE)
    add_wp(f'STEP3_Lift_Pod{pod_id}', lift_pose, 'closed', 0.05)
    above_pan = Pose(pan.x, pan.y, pan.z+0.100+SAFE_Z_CLEARANCE)
    add_wp(f'STEP3b_Transit_Pan{pan_id}', above_pan, 'closed', 0.10)
    add_wp(f'STEP4_Dispense_Pod{pod_id}_Pan{pan_id}',
           Pose(pan.x, pan.y, pan.z+0.100, pitch=math.pi), 'closed', 0.02)
    add_wp('STEP5a_Rise', above_pan, 'closed', 0.05)
    add_wp(f'STEP5b_Return_Rack', lift_pose, 'closed', 0.08)
    add_wp(f'STEP5c_Place_Pod{pod_id}', grip_pose, 'open', 0.03)
    add_wp('STEP5d_Retreat',
           Pose(pod.x, pod.y-0.080, pod.z+SPICE_POD_LIFT), 'open', 0.05)
    wps.append(Waypoint('HOME_RETURN', Pose(*home_pos), home_j.copy(), 'open'))
    print(f'\n  📊 Total waypoints: {len(wps)}')
    return wps

path_p1_pan2  = plan_micro_dispense(pod_id=1,  pan_id=2)
path_p19_pan1 = plan_micro_dispense(pod_id=19, pan_id=1)

## 9. Task 2 Visualization

In [ ]:
fig = plt.figure(figsize=(16,7))
fig.patch.set_facecolor('#1a1a2e')

ax1 = fig.add_subplot(121, projection='3d')
ax1.set_facecolor('#16213e')
ax1.set_title('Task 2: Micro Dispense — 3D Path', color='white', fontsize=12, fontweight='bold')

# Draw pods
for pid, p in SPICE_PODS.items():
    col = '#e91e63' if pid <= 10 else '#9c27b0'
    theta = np.linspace(0,2*np.pi,20)
    z_c = np.array([p.z, p.z+0.06])
    th_g, z_g = np.meshgrid(theta, z_c)
    ax1.plot_surface(p.x+0.018*np.cos(th_g), p.y+0.018*np.sin(th_g), z_g,
                    color=col, alpha=0.5)
    ax1.text(p.x, p.y, p.z+0.08, str(pid), fontsize=4, ha='center', color='white')

# Draw pans
for pid, pp in PAN_CENTERS.items():
    theta = np.linspace(0,2*np.pi,30)
    z_c = np.array([pp.z, pp.z+0.06])
    th_g, z_g = np.meshgrid(theta, z_c)
    ax1.plot_surface(pp.x+0.13*np.cos(th_g), pp.y+0.13*np.sin(th_g), z_g,
                    color='#4fc3f7', alpha=0.3)
    ax1.text(pp.x, pp.y, pp.z+0.09, f'Pan{pid}', color='#4fc3f7', fontsize=8, ha='center')

pts_p1  = np.array([[w.pose.x,w.pose.y,w.pose.z] for w in path_p1_pan2])
pts_p19 = np.array([[w.pose.x,w.pose.y,w.pose.z] for w in path_p19_pan1])
ax1.plot(pts_p1[:,0], pts_p1[:,1], pts_p1[:,2], 'r-o', lw=2, ms=4, label='Pod1→Pan2', alpha=0.9)
ax1.plot(pts_p19[:,0],pts_p19[:,1],pts_p19[:,2], 'y-o', lw=2, ms=4, label='Pod19→Pan1',alpha=0.9)
ax1.scatter([0],[0],[0], s=150, c='lime', marker='^')

ax1.set_xlabel('X (m)',color='white',fontsize=8)
ax1.set_ylabel('Y (m)',color='white',fontsize=8)
ax1.set_zlabel('Z (m)',color='white',fontsize=8)
ax1.tick_params(colors='white', labelsize=6)
ax1.xaxis.pane.fill = ax1.yaxis.pane.fill = ax1.zaxis.pane.fill = False
ax1.legend(fontsize=9, labelcolor='white', facecolor='#1a1a2e', framealpha=0.6)

# Side view (XZ)
ax2 = fig.add_subplot(122)
ax2.set_facecolor('#16213e')
ax2.set_title('Task 2: Side View — Pod Rack (XZ)', color='white', fontsize=12, fontweight='bold')
for pid, p in SPICE_PODS.items():
    col = '#e91e63' if pid <= 10 else '#9c27b0'
    c = plt.Circle((p.x, p.z), 0.018, color=col, alpha=0.7)
    ax2.add_patch(c)
    ax2.text(p.x, p.z+0.03, str(pid), ha='center', fontsize=5, color='white')
ax2.plot(pts_p1[:,0], pts_p1[:,2], 'r-o', lw=2, ms=4, label='Pod1→Pan2')
ax2.plot(pts_p19[:,0],pts_p19[:,2],'y-o', lw=2, ms=4, label='Pod19→Pan1')
ax2.set_xlabel('X (m)',color='white'); ax2.set_ylabel('Z (m)',color='white')
ax2.tick_params(colors='white')
ax2.legend(fontsize=9, facecolor='#1a1a2e', labelcolor='white', framealpha=0.7)
ax2.grid(True, color='gray', alpha=0.2)
ax2.set_xlim(0.0, 0.5); ax2.set_ylim(0.7, 1.7)
for sp in ax2.spines.values(): sp.set_edgecolor('gray')

plt.tight_layout()
plt.savefig('task2_micro_path.png', dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()
print('✅ Task 2 visualization complete')

## 10. Test Cases & Validation

In [ ]:
print('='*60)
print('TEST CASES')
print('='*60)

tests = [
    ('FK at home (verify EE position)',
     lambda: PiperKinematics.forward_kinematics([0.0,0.3,-0.5,0.0,0.4,0.0])[0]),
    ('IK for Pan2 approach',
     lambda: PiperKinematics.inverse_kinematics(
         Pose(PAN_CENTERS[2].x, PAN_CENTERS[2].y, PAN_CENTERS[2].z+0.15))[1]),
    ('Container 5 reachability check',
     lambda: PiperKinematics.check_reachability(MACRO_CONTAINERS[5])),
    ('Collision: direct C5→P2 (expect blocked)',
     lambda: CollisionChecker().check_path(MACRO_CONTAINERS[5].to_array(), PAN_CENTERS[2].to_array())),
    ('Joint limit clamping (out-of-range values)',
     lambda: PiperKinematics._clamp([10.,-10.,10.,-10.,10.,-10.])),
    ('All 20 pods reachable (any True)',
     lambda: any(PiperKinematics.check_reachability(SPICE_PODS[i]) for i in range(1,21))),
    ('Macro path C5→P2: no joint violations',
     lambda: len([w for w in path_c5_p2 if any(
         not (lo<=a<=hi)
         for a,(lo,hi) in zip(w.joint_angles,PiperKinematics.JOINT_LIMITS.values())
     )]) == 0),
    ('Micro path Pod1→Pan2 waypoint count',
     lambda: len(path_p1_pan2)),
]

for name, fn in tests:
    try:
        result = fn()
        print(f'  ✅ {name}')
        print(f'     → {result}')
    except Exception as e:
        print(f'  ❌ {name}: {e}')

## 11. Export Results to JSON

In [ ]:
def export_path(path, label):
    return [
        {'step': wp.name,
         'position_m': [round(wp.pose.x,4), round(wp.pose.y,4), round(wp.pose.z,4)],
         'joint_angles_deg': [round(math.degrees(a),2) for a in wp.joint_angles],
         'gripper': wp.gripper_state,
         'speed_ms': wp.speed}
        for wp in path
    ]

results = {
    'robot': 'AgileX Piper 6-DOF',
    'candidate': 'Rachit — SRM IST Kattankulathur',
    'task_1_macro_dispense': {
        'sequence_1': {'Container_5_to_Pan_2': export_path(path_c5_p2, 'C5→P2')},
        'sequence_2': {'Container_1_to_Pan_1': export_path(path_c1_p1, 'C1→P1')},
    },
    'task_2_micro_dispense': {
        'sequence_1': {'Pod_1_to_Pan_2':  export_path(path_p1_pan2,  'P1→P2')},
        'sequence_2': {'Pod_19_to_Pan_1': export_path(path_p19_pan1, 'P19→P1')},
    }
}

with open('posha_path_planning_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print('✅ Results exported to posha_path_planning_results.json')
print(f'   Task 1 C5→P2: {len(path_c5_p2)} waypoints')
print(f'   Task 1 C1→P1: {len(path_c1_p1)} waypoints')
print(f'   Task 2 Pod1→Pan2:  {len(path_p1_pan2)} waypoints')
print(f'   Task 2 Pod19→Pan1: {len(path_p19_pan1)} waypoints')

## 12. Hardware Implementation Recommendations

1. **Mount robot base** at assembly origin with 6-bolt flange (coordinates confirmed from STEP file)
2. **Force-torque sensor** at wrist joint for container gripping feedback (1.2kg load, 1.5kg capacity)
3. **RGB-D camera** for container pose correction before gripping
4. **Soft real-time control loop** at 100Hz for smooth joint interpolation
5. **Thermal shielding** on arm links nearest to stove heating element
6. **Emergency stop** triggered if force exceeds 150% of expected load
7. **Gripper pressure feedback** for spice pod grip confirmation (fragile pods)

---
*POSHA Robotics Engineering Internship Assignment | Rachit | April 2026*